In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# 1. Load the Dataset
df = pd.read_csv('AB_NYC_2019.csv')

# 2. Data Cleaning & Handling Missing Values
# Drop columns with high cardinality/text data that offer little numerical predictive value
columns_to_drop = ['id', 'name', 'host_name', 'last_review']
df = df.drop(columns=columns_to_drop)

# Missing values in 'reviews_per_month' correlate with listings having 0 reviews.
df['reviews_per_month'] = df['reviews_per_month'].fillna(0)

# 3. Handling Outliers
# Price = 0 is invalid for rentals. Prices > $1,000 are extreme outliers (top ~1%) that skew models.
df = df[(df['price'] > 0) & (df['price'] <= 1000)].copy()

# 4. Feature Engineering & Transformation
# Target Variable Transformation: Airbnb prices are heavily right-skewed. 
# Applying a log transformation normalizes the distribution for better regression performance.
df['log_price'] = np.log1p(df['price'])

# 5. Feature Selection and Preprocessing
# We drop 'neighbourhood' due to high cardinality (221 unique values), relying on 
# 'neighbourhood_group' (Borough) and exact latitude/longitude coordinates instead.
X = df.drop(columns=['price', 'log_price', 'neighbourhood', 'host_id'])
y = df['log_price']

# Define feature types
numeric_features = ['latitude', 'longitude', 'minimum_nights', 'number_of_reviews', 
                    'reviews_per_month', 'calculated_host_listings_count', 'availability_365']
categorical_features = ['neighbourhood_group', 'room_type']

# Create preprocessing pipelines
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Split the data before fitting the preprocessor to prevent data leakage
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit and transform the data
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Convert back to DataFrame for visibility (Optional)
feature_names = (preprocessor.named_transformers_['num'].get_feature_names_out().tolist() + 
                 preprocessor.named_transformers_['cat'].get_feature_names_out().tolist())
X_train_df = pd.DataFrame(X_train_processed, columns=feature_names)

In [2]:
#import numpy as np
#import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Models
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# 1. Load and Prepare Cleaned Data
df = pd.read_csv('AB_NYC_2019.csv')
df = df.drop(columns=['id', 'name', 'host_name', 'last_review'])
df['reviews_per_month'] = df['reviews_per_month'].fillna(0)
df = df[(df['price'] > 0) & (df['price'] <= 1000)].copy()
df['log_price'] = np.log1p(df['price'])

X = df.drop(columns=['price', 'log_price', 'neighbourhood', 'host_id'])
y = df['log_price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2. Preprocessing Pipeline
numeric_features = [
    'latitude', 'longitude', 'minimum_nights', 'number_of_reviews', 
    'reviews_per_month', 'calculated_host_listings_count', 'availability_365'
]
categorical_features = ['neighbourhood_group', 'room_type']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

# 3. Model Comparison
models = {
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
}

def evaluate_predictions(y_true_log, y_pred_log):
    """Calculates metrics on both log and actual dollar scale."""
    y_true_dollar = np.expm1(y_true_log)
    y_pred_dollar = np.expm1(y_pred_log)
    
    rmse_dollar = np.sqrt(mean_squared_error(y_true_dollar, y_pred_dollar))
    mae_dollar = mean_absolute_error(y_true_dollar, y_pred_dollar)
    r2 = r2_score(y_true_dollar, y_pred_dollar)
    return rmse_dollar, mae_dollar, r2

print("=== Baseline Model Comparison (Test Set - Dollar Scale) ===")
for name, model in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    rmse, mae, r2 = evaluate_predictions(y_test, preds)
    print(f"{name:20s} | RMSE: ${rmse:.2f} | MAE: ${mae:.2f} | R²: {r2:.4f}")

# 4. Hyperparameter Tuning (Gradient Boosting Regressor)
# Build an end-to-end pipeline with the top-performing candidate
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', GradientBoostingRegressor(random_state=42))
])

param_distributions = {
    'regressor__n_estimators': [100, 150, 200],
    'regressor__learning_rate': [0.03, 0.05, 0.1],
    'regressor__max_depth': [4, 6, 8],
    'regressor__min_samples_split': [10, 20, 30],
    'regressor__subsample': [0.8, 1.0]
}

random_search = RandomizedSearchCV(
    estimator=full_pipeline,
    param_distributions=param_distributions,
    n_iter=10,
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)
best_pipeline = random_search.best_estimator_

# 5. Overfitting Check and Final Model Evaluation
train_preds = best_pipeline.predict(X_train)
test_preds = best_pipeline.predict(X_test)

train_rmse, train_mae, train_r2 = evaluate_predictions(y_train, train_preds)
test_rmse, test_mae, test_r2 = evaluate_predictions(y_test, test_preds)

print("\n=== Final Model Performance Check ===")
print(f"Train Set -> RMSE: ${train_rmse:.2f} | MAE: ${train_mae:.2f} | R²: {train_r2:.4f}")
print(f"Test Set  -> RMSE: ${test_rmse:.2f} | MAE: ${test_mae:.2f} | R²: {test_r2:.4f}")

# 6. Save the Full End-to-End Pipeline
joblib.dump(best_pipeline, 'airbnb_price_pipeline.joblib')
print("\nFinal pipeline successfully saved to 'airbnb_price_pipeline.joblib'")

=== Baseline Model Comparison (Test Set - Dollar Scale) ===
Ridge Regression     | RMSE: $98.76 | MAE: $52.40 | R²: 0.3009
Random Forest        | RMSE: $88.72 | MAE: $45.70 | R²: 0.4359
Gradient Boosting    | RMSE: $90.01 | MAE: $46.34 | R²: 0.4193
Fitting 3 folds for each of 10 candidates, totalling 30 fits

=== Final Model Performance Check ===
Train Set -> RMSE: $77.75 | MAE: $39.70 | R²: 0.5537
Test Set  -> RMSE: $87.84 | MAE: $45.23 | R²: 0.4470

Final pipeline successfully saved to 'airbnb_price_pipeline.joblib'
